# Advanced agents — MCP (Model Context Protocol)
Connect an agent to tools that live **outside** the code, inside an MCP server: first a **local** server (stdio), then an **online** one. The agent loads the server's tools and its system prompt, then answers using them.

Only a Gemini key is needed (from `.env`, Colab Secrets or pasted below): `GOOGLE_API_KEY` — the local project keeps the same key as `GEMINI_API_KEY`, both are accepted. Same small, cheap model as the Module 1 notebooks: `gemini-3.1-flash-lite`.

> **Model fix:** the original lesson used an OpenAI model (`gpt-5-nano`) which needs an OpenAI key. Like every notebook in this path, this one uses Gemini — no extra key, no extra cost.

**Idea:** MCP (Model Context Protocol) is an open standard that decouples tools from agents. One server — a plain Python script — declares three kinds of content: **tools**, **resources** and **prompts** — and any agent on any machine can consume them over `stdio` or HTTP. In lesson 1_2 the tools were defined inside the notebook; here they are defined **once, in a server**, and the agent just connects.

---

## What I learned (notes for anyone starting)
1. **MCP separates the tool from the agent.** The notebook does not contain the tool code: it asks a server for it (`get_tools`). Change the server, and the agent's abilities change without touching the agent.
2. **`MultiServerMCPClient`** connects one agent to several MCP servers, each started by a name (`"local_server"`, `"time"`, ...).
3. **Three kinds of content:** tools (callable), resources (readable data behind a URI), prompts (reusable system prompts). `get_prompt(server, name)` loads a system prompt straight from the server.
4. **The rest is the same as Module 1:** `create_agent` with a model, tools and a system prompt — plus a **thread id** and a checkpointer so follow-up questions still work (lesson 1_3).
5. **stdio transport = a subprocess.** The client launches the server as a child process; the Windows fixes cell below is what makes this work in Jupyter on Windows.
6. **Wrong paths fail first.** The most common failure is a wrong path to the server file in the client configuration — the instructor warned us, believe him.

## Mistakes I made (so you can avoid them)
- The MCP packages are **not** installed by default: install `langchain-mcp-adapters` and `mcp-server-time` (see Setup).
- On Windows, the stdio server silently fails without the **Proactor event loop** — hence the Windows fixes cell.
- There is no `get_prompts` in the current package: it is `get_prompt(server, name)` (singular).
- `MultiServerMCPClient` **cannot** be used with `async with`: it connects lazily on the first `get_tools()`. The child processes end when the kernel stops.
- `command: "python"` can start the wrong Python when Jupyter uses a virtual environment: use `sys.executable` (the Python running the notebook).
- Watch the free-tier quota on Gemini: a 429 error is Google-side (quota), not a bug in your agent.

## Setup

In [1]:
import importlib.util as _ilu
import subprocess as _sp
import sys as _sys

# Install only what is missing. Uses pip when available, falls back to uv
# (the local venv on this machine is created by uv and has no pip).
REQUIRED = [
    "langchain",
    "langchain_google_genai",
    "langchain_mcp_adapters",
    "mcp_server_time",
    "dotenv",
]
missing = [pkg for pkg in REQUIRED if _ilu.find_spec(pkg) is None]

if missing:
    print("Installing:", ", ".join(missing))
    if _sp.run([_sys.executable, "-m", "pip", "install", "-q", *missing]).returncode != 0:
        _sp.run(["uv", "pip", "install", *missing])
else:
    print("All packages already installed.")

All packages already installed.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # local run: finds the .env above the notebook folder

# Colab run: read the key from the Secrets panel (key icon in the left sidebar)
try:
    from google.colab import userdata
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
except ImportError:
    pass

def is_real(value):
    # empty values and template placeholders ("your_..." ) count as missing
    return bool(value) and not value.startswith("your_")

# Accept both names: the local project keeps the key as GEMINI_API_KEY
for name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
    if is_real(os.getenv(name)):
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = os.getenv(name)
        break

# Still missing? Paste it here (hidden input).
if not is_real(os.getenv("GOOGLE_API_KEY")):
    try:
        value = getpass("Gemini API key is missing - paste it (Enter to skip): ").strip()
    except Exception:
        value = ""
    if value:
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = value

print("GOOGLE_API_KEY:", "set" if is_real(os.getenv("GOOGLE_API_KEY")) else "MISSING")

# Tracing with a placeholder LangSmith key only produces 401/403 warnings: switch it off
if not is_real(os.getenv("LANGSMITH_API_KEY")):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: off (no valid LANGSMITH_API_KEY)")

GOOGLE_API_KEY: set
LangSmith tracing: off (no valid LANGSMITH_API_KEY)


In [3]:
import sys
import asyncio

# Fixes for Windows issues in Jupyter notebooks (launching MCP stdio servers)
if sys.platform == "win32":
    # 1. Use the ProactorEventLoop policy, which supports subprocesses
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid the fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

print("Windows fixes applied:", sys.platform)

Windows fixes applied: win32


## 1. Local MCP server (stdio)
An MCP server is a plain Python script. Ours (`resources/2_1_mcp_server.py`) exposes a **tool** (`add`), a **resource** (`greeting://{name}`) and a **prompt** (named `prompt`). The client starts it as a subprocess over the `stdio` transport.

> **`MultiServerMCPClient({name: {transport, command, args}})`**: connects to one or more MCP servers. `command` + `args` say how to launch each server.

In [4]:
import os
import sys

# Find the server script even if the notebook is opened from another folder
def find_server():
    here = os.getcwd()
    for _ in range(5):
        candidate = os.path.join(here, "resources", "2_1_mcp_server.py")
        if os.path.exists(candidate):
            return candidate
        here = os.path.dirname(here)
    return None

local_server = find_server()
assert local_server is not None, "resources/2_1_mcp_server.py not found - wrong working folder?"
print("MCP server script:", local_server)

from langchain_mcp_adapters.client import MultiServerMCPClient

# One client, several servers. Each server has a name.
client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": sys.executable,   # the same Python running this notebook
            "args": [local_server],
        }
    }
)

MCP server script: C:\Users\Kassouar Fatima\Downloads\LangChain\Module 2 - Advanced Agent\notebooks\resources\2_1_mcp_server.py


In [5]:
# Ask the server for its three kinds of content
tools = await client.get_tools()
resources = await client.get_resources("local_server")
prompt = await client.get_prompt("local_server", "prompt")
system_prompt = prompt[0].content

print("Tools exposed by the server:")
for t in tools:
    print(f"- {t.name}: {t.description}")

print("\nResources:", resources)   # templates exist but are not listed by get_resources

print("\nSystem prompt loaded from the server:\n", system_prompt)

Tools exposed by the server:
- add: Add two integers together. Use it for any arithmetic the user asks for.

Resources: []

System prompt loaded from the server:
 You are a helpful assistant. Keep answers short and clear, and only use the tools you have when they are needed.


The prompt came **from the server**, not from the notebook. Now the agent is built exactly like in Module 1 — same `create_agent`, plus `checkpointer=InMemorySaver()` and the same **thread id** so follow-up questions still work (lesson 1_3).

In [6]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

MODEL = "google_genai:gemini-3.1-flash-lite"  # same small, cheap model as Module 1

agent = create_agent(
    model=MODEL,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),   # memory: snapshots saved per thread id (lesson 1_3)
)

config = {"configurable": {"thread_id": "1"}}
print("agent ready, MODEL =", MODEL)

agent ready, MODEL = google_genai:gemini-3.1-flash-lite


In [7]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Using the add tool, what is 40 + 2?")]},
    config=config,
)

print(response["messages"][-1].text)
print("\nTool calls made by the agent:")
for m in response["messages"]:
    if getattr(m, "tool_calls", None):
        print(" -", [tc["name"] for tc in m.tool_calls])

40 + 2 is 42.

Tool calls made by the agent:
 - ['add']


## 2. Online MCP server
`mcp-server-time` is a published MCP server that knows the current time and can convert between timezones. It is launched the same way as the local one — with the module's entry point (`python -m mcp_server_time`) and the *Africa/Algiers* timezone.

> **Same protocol, different server:** only the `command`/`args` change. This is the whole point of MCP — the agent code stays the same.

In [8]:
time_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": sys.executable,   # environment where mcp-server-time is installed
            "args": ["-m", "mcp_server_time", "--local-timezone=Africa/Algiers"],
        }
    }
)

time_tools = await time_client.get_tools()

for t in time_tools:
    print(f"- {t.name}: {t.description}")

- get_current_time: Get current time in a specific timezone
- convert_time: Convert time between timezones


In [9]:
time_agent = create_agent(model=MODEL, tools=time_tools)

answer = await time_agent.ainvoke(
    {"messages": [HumanMessage(content="What time is it?")]}
)

print(answer["messages"][-1].text)
print("\nTool calls made by the agent:")
for m in answer["messages"]:
    if getattr(m, "tool_calls", None):
        print(" -", [tc["name"] for tc in m.tool_calls])

It is 6:27 PM on Friday, September 25, 2026.

Tool calls made by the agent:
 - ['get_current_time']


## Summary of functions

| Function / class | Simple definition |
|---|---|
| `MultiServerMCPClient({name: {transport, command, args}})` | Connects one agent to several MCP servers, each started by a name. |
| `await client.get_tools()` | Asks the servers for their tools; starts the connections lazily. |
| `await client.get_resources(server)` | Asks a server for its readable resources (templates exist but are not listed). |
| `await client.get_prompt(server, name)` | Loads a system prompt defined on a server (singular form). |
| `@mcp.tool()` | Tells FastMCP that the function below is a tool. |
| `@mcp.resource("uri://...")` | Tells FastMCP that the function is readable data behind a URI. |
| `@mcp.prompt()` | Tells FastMCP that the function builds a reusable system prompt. |
| `create_agent(model, tools=..., system_prompt=...)` | Builds the agent around a model with the server's tools and prompt. |
| `create_agent(..., checkpointer=InMemorySaver())` | Adds memory (lesson 1_3): a snapshot of the state is saved under each thread id. |
| `await agent.ainvoke(..., config={"configurable": {"thread_id": "1"}})` | Runs the agent; the thread id keeps state for follow-up questions. |
| `sys.executable` | The same Python running the notebook — guarantees the MCP subprocesses use the right interpreter. |
| Proactor event loop | Windows loop policy required by MCP stdio subprocesses. |

By
> Kassouar Fatima                   25-09-2026